# Portfolio Optimization with Constraint-Preserving QAOA

This notebook investigates portfolio optimization using the **Quantum Approximate Optimization Algorithm (QAOA)** with a constraint-preserving XY mixer.

The objective is to select exactly \(K\) assets from a universe of \(n\) assets while maximizing the risk-adjusted portfolio objective.

The experiments are conducted using noiseless statevector simulation in Qiskit.

**Main objectives:**
- Establish exact and classical benchmark solutions.
- Implement QAOA with a warm-start initial state and an XY mixer.
- Analyze the effect of circuit depth \(p\).
- Evaluate solution quality, optimal-solution probability and computational cost.
- Compare QAOA against classical baselines.

In [ ]:

import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from itertools import combinations
from math import comb

from scipy.optimize import minimize

from qiskit.circuit.library import QAOAAnsatz
from qiskit.primitives import StatevectorEstimator, StatevectorSampler
from qiskit_optimization import QuadraticProgram
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector
from qiskit.circuit import Parameter
from qiskit.circuit.library import XXPlusYYGate
import time


# ============================================================
# 0. CONFIGURATION
# ============================================================

# ============================================================
# DATOS INTERMEDIATE
# ============================================================

DATA_DIR = Path(
    r"C:\Users\fio\Documents\qff\quantum-optimization-qml-qff2026\data\intermediate"
)

# Si solo hay un JSON, lo detecta automáticamente.
json_files = sorted(DATA_DIR.glob("*.json"))

if len(json_files) != 1:
    raise RuntimeError(
        f"Se esperaba un único JSON en {DATA_DIR}. "
        f"Archivos encontrados: {[f.name for f in json_files]}"
    )

DATA_PATH = json_files[0]

OUTPUT_DIR = DATA_DIR / "results_intermediate_xy"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Solo se utiliza si K no viene incluido en el JSON.
K_DEFAULT = 6

with DATA_PATH.open("r", encoding="utf-8") as file:
    data = json.load(file)

n = int(data["n_activos"])
tickers = data["tickers"]
mu = np.asarray(data["mu"], dtype=float)
sigma = np.asarray(data["sigma"], dtype=float)

K = int(data.get("K", data.get("k_activos", K_DEFAULT)))
Q_RISK = float(data.get("q", data.get("q_risk", 0.5)))

print("Intermediate assets:", n)
print("Assets to select:", K)
print("Data file:", DATA_PATH.name)

# ============================================================
# 0.1 CONFIGURATION
# ============================================================


P_MAIN = 2
MAIN_SEED = 42

FAST_MODE = False

if FAST_MODE:
    # Quick screening
    DEPTHS = [1, 2, 3]
    RESTARTS = 1
    SHOTS = 512
    MAX_ITER = 60
    EXPERIMENT_SEEDS = [42, 0, 11]

else:
    # Final validation
    DEPTHS = [1, 2, 3]
    RESTARTS = 2
    SHOTS = 4096
    MAX_ITER = 150
    EXPERIMENT_SEEDS = [
        0, 11, 22, 33, 42, 51,
        64, 73, 85, 96, 107
    ]




# ============================================================
# 1. LOAD AND VALIDATE FINANCIAL DATA
# ============================================================


if len(tickers) != n:
    raise ValueError("The number of tickers does not match n_activos.")
if mu.shape != (n,):
    raise ValueError(f"mu must have shape ({n},), got {mu.shape}.")
if sigma.shape != (n, n):
    raise ValueError(
        f"sigma must have shape ({n}, {n}), got {sigma.shape}."
    )
if not (0 <= K <= n):
    raise ValueError("K must be between 0 and n.")
if not np.all(np.isfinite(mu)) or not np.all(np.isfinite(sigma)):
    raise ValueError("Financial data contains NaN or infinite values.")
if not np.allclose(sigma, sigma.T, atol=1e-10):
    raise ValueError("The covariance matrix is not symmetric.")

# Remove tiny numerical asymmetries.
sigma = (sigma + sigma.T) / 2.0

eigenvalues = np.linalg.eigvalsh(sigma)
print("\nMinimum covariance eigenvalue:", float(eigenvalues.min()))
if eigenvalues.min() < -1e-8:
    raise ValueError("The covariance matrix is not positive semidefinite.")

print("Financial data validation passed.")


# ============================================================
# 2. FINANCIAL OBJECTIVE
# ============================================================

def portfolio_objective(x, mu_vector=mu, covariance=sigma, risk_aversion=Q_RISK):
    """Return f(x) = mu.T @ x - q * x.T @ sigma @ x."""
    x = np.asarray(x, dtype=float)
    expected_return = float(mu_vector @ x)
    portfolio_risk = float(x @ covariance @ x)
    return expected_return - risk_aversion * portfolio_risk


# ============================================================
# 2.5. DIAGNOSTIC: TEST DIFFERENT CARDINALITY VALUES
# ============================================================

def greedy_for_k(k):
    """Greedy heuristic for a specified cardinality."""
    x = np.zeros(n, dtype=int)

    for _ in range(k):
        best_i = None
        best_value = -np.inf

        for i in range(n):
            if x[i] == 0:
                candidate = x.copy()
                candidate[i] = 1
                value = portfolio_objective(candidate)

                if value > best_value:
                    best_value = value
                    best_i = i

        x[best_i] = 1

    return x


def exact_for_k(k):
    """Exact optimum among portfolios containing exactly k assets."""
    combos_k = list(combinations(range(n), k))

    X = np.zeros((len(combos_k), n), dtype=int)

    for row, combo in enumerate(combos_k):
        X[row, list(combo)] = 1

    values = (
        X @ mu
        - Q_RISK * np.einsum(
            "bi,ij,bj->b",
            X,
            sigma,
            X
        )
    )

    idx = int(np.argmax(values))

    return X[idx], float(values[idx])


results_k = []

for k in range(2, n - 1):

    greedy_candidate = greedy_for_k(k)
    greedy_val = portfolio_objective(greedy_candidate)

    exact_x_k, exact_val_k = exact_for_k(k)

    results_k.append({
        "K": k,
        "feasible_portfolios": comb(n, k),
        "feasible_percentage": (
            100 * comb(n, k) / (2 ** n)
        ),
        "greedy_objective": greedy_val,
        "exact_objective": exact_val_k,
        "greedy_gap": exact_val_k - greedy_val,
        "greedy_ratio": (
            greedy_val / exact_val_k
            if exact_val_k > 0 else np.nan
        )
    })


k_df = pd.DataFrame(results_k)

print("\nCARDINALITY SENSITIVITY ANALYSIS")
print(k_df.to_string(index=False))

k_df.to_csv(
    OUTPUT_DIR / "cardinality_sensitivity.csv",
    index=False
)




# ============================================================
# 3. EXACT BRUTE-FORCE BENCHMARK
# ============================================================


# Enumerar únicamente carteras con exactamente K activos.
combos = np.asarray(
    list(combinations(range(n), K)),
    dtype=int
)

feasible_x = np.zeros((len(combos), n), dtype=int)

if K > 0:
    feasible_x[np.arange(len(combos))[:, None], combos] = 1

# Evaluar todas las carteras factibles.
feasible_values = (
    feasible_x @ mu
    - Q_RISK * np.einsum(
        "bi,ij,bj->b",
        feasible_x,
        sigma,
        feasible_x
    )
)

best_idx = int(np.argmax(feasible_values))

optimal_x = feasible_x[best_idx]
optimal_value = float(feasible_values[best_idx])

optimal_solutions = feasible_x[
    np.isclose(
        feasible_values,
        optimal_value,
        rtol=1e-9,
        atol=1e-10
    )
]

print("Feasible portfolios:", len(feasible_x))
print("Exact optimum:", optimal_value)
print(
    "Optimal assets:",
    [tickers[i] for i in range(n) if optimal_x[i] == 1]
)

# ----------------------------------------
# Define greedy and local search baselines for comparison.
# ----------------------------------------

def greedy_baseline():
    """Selecciona sucesivamente el activo con mejor valor marginal."""
    x = np.zeros(n, dtype=int)

    for _ in range(K):
        best_i = None
        best_value = -np.inf

        for i in range(n):
            if x[i] == 0:
                candidate = x.copy()
                candidate[i] = 1

                value = portfolio_objective(candidate)

                if value > best_value:
                    best_value = value
                    best_i = i

        x[best_i] = 1

    return x




def local_search_baseline(x):
    """Mejora la cartera mediante intercambios 1 por 1."""
    x = np.asarray(x, dtype=int).copy()

    improved = True

    while improved:
        improved = False
        best_value = portfolio_objective(x)
        best_candidate = None

        selected = np.where(x == 1)[0]
        unselected = np.where(x == 0)[0]

        for i in selected:
            for j in unselected:
                candidate = x.copy()
                candidate[i] = 0
                candidate[j] = 1

                value = portfolio_objective(candidate)

                if value > best_value + 1e-12:
                    best_value = value
                    best_candidate = candidate

        if best_candidate is not None:
            x = best_candidate
            improved = True

    return x


# ============================================================
# FINAL CARDINALITY DIAGNOSTIC
# ============================================================

results_k = []

for k in range(2, n - 1):

    greedy_candidate = greedy_for_k(k)
    local_candidate = local_search_baseline(greedy_candidate)
    exact_candidate, exact_val = exact_for_k(k)

    greedy_val = portfolio_objective(greedy_candidate)
    local_val_k = portfolio_objective(local_candidate)

    results_k.append({
        "K": k,
        "feasible_portfolios": comb(n, k),
        "feasible_percentage": 100 * comb(n, k) / (2**n),
        "greedy_value": greedy_val,
        "local_search_value": local_val_k,
        "exact_value": exact_val,
        "greedy_gap": exact_val - greedy_val,
        "local_search_gap": exact_val - local_val_k,
        "greedy_ratio": greedy_val / exact_val if exact_val > 0 else np.nan,
        "local_search_ratio": local_val_k / exact_val if exact_val > 0 else np.nan
    })

k_df = pd.DataFrame(results_k)

print(k_df.to_string(index=False))

k_df.to_csv(
    OUTPUT_DIR / "cardinality_sensitivity.csv",
    index=False
)


greedy_x = greedy_baseline()
local_x = local_search_baseline(greedy_x)

greedy_value = portfolio_objective(greedy_x)
local_value = portfolio_objective(local_x)

print("Greedy objective:", greedy_value)
print("Local search objective:", local_value)
print("Exact optimum:", optimal_value)


# ============================================================
# 4. BUILD UNCONSTRAINED FINANCIAL HAMILTONIAN
# ============================================================

qp_xy = QuadraticProgram()

for ticker in tickers:
    qp_xy.binary_var(name=ticker)

# Objective only: no cardinality penalty or equality constraint.
qp_xy.maximize(
    linear=mu,
    quadratic=-Q_RISK * sigma
)

H_xy, offset_xy = qp_xy.to_ising()

print("\nXY financial Hamiltonian:")
print(H_xy)
print("Offset:", offset_xy)



# ============================================================
# 5. INITIAL STATE AND CONSTRAINT-PRESERVING XY MIXER
# ============================================================

# Warm-start from GREEDY.
# We keep local search as a separate, stronger classical baseline.
initial_state = QuantumCircuit(n)

for i in np.where(greedy_x == 1)[0]:
    initial_state.x(int(i))

assert int(greedy_x.sum()) == K

# Parameter shared by all ring interactions in one mixer layer.
xy_beta = Parameter("beta_xy")

# Build a ring of exact excitation-preserving interactions.
xy_mixer = QuantumCircuit(n, name="XY_ring_mixer")

for i in range(n):
    j = (i + 1) % n

    xy_mixer.append(
        XXPlusYYGate(xy_beta, beta=0.0),
        [i, j]
    )

# Verify cardinality preservation on the initial state.
probe = initial_state.compose(
    xy_mixer.assign_parameters({xy_beta: 0.37})
)

probe_state = Statevector.from_instruction(probe)

leakage = sum(
    probability
    for bitstring, probability in probe_state.probabilities_dict().items()
    if bitstring.count("1") != K
)

print("Initial cardinality:", int(greedy_x.sum()))
print("XY mixer leakage probability:", leakage)

if leakage > 1e-10:
    raise RuntimeError("The XY mixer does not preserve cardinality.")

print("XY mixer validation passed.")


# Normalize the cost Hamiltonian for numerical stability.
H_scale = float(np.max(np.abs(H_xy.coeffs)))

if H_scale <= 0:
    raise ValueError("The cost Hamiltonian has zero coefficients.")

H_cost = H_xy * (1.0 / H_scale)
offset_cost = offset_xy / H_scale



# ============================================================
# 6. BITSTRING DECODING AND METRICS
# ============================================================

def decode_bitstring(key, num_qubits):
    bits = key.replace(" ", "")[::-1]

    if len(bits) != num_qubits:
        raise ValueError(
            f"Expected {num_qubits} bits, received {len(bits)}"
        )

    return np.asarray([int(bit) for bit in bits], dtype=int)


def approximation_ratio(value):
    if optimal_value <= 0:
        return np.nan

    return float(value) / optimal_value


# ============================================================
# 7. CONSTRAINT-PRESERVING QAOA EXECUTION
# ============================================================

ESTIMATOR = StatevectorEstimator()


def run_xy_qaoa(
    p,
    seed=42,
    shots=SHOTS,
    maxiter=MAX_ITER,
    restarts=RESTARTS
):
    if p < 1 or shots < 1 or maxiter < 1 or restarts < 1:
        raise ValueError("Invalid QAOA configuration.")

    start_time = time.perf_counter()

    ansatz_no_meas = QAOAAnsatz(
        cost_operator=H_cost,
        reps=p,
        initial_state=initial_state,
        mixer_operator=xy_mixer,
        flatten=True
    )

    ansatz_meas = ansatz_no_meas.copy()
    ansatz_meas.measure_all()

    def cost_function(params):
        pub = (ansatz_no_meas, H_cost, params)

        energy = ESTIMATOR.run([pub]).result()[0].data.evs

        return float(np.asarray(energy).item())

    rng = np.random.default_rng(seed)

    best_optimization = None

    # Multiple COBYLA restarts.
    for restart in range(restarts):

        x0 = rng.uniform(
            -np.pi,
            np.pi,
            ansatz_no_meas.num_parameters
        )

        optimization = minimize(
            cost_function,
            x0,
            method="COBYLA",
            options={
                "maxiter": maxiter,
                "rhobeg": 0.5,
                "tol": 1e-2
            }
        )

        if (
            best_optimization is None
            or optimization.fun < best_optimization.fun
        ):
            best_optimization = optimization

    # Sample the best optimized circuit.
    sampler = StatevectorSampler(seed=seed)

    samples = sampler.run(
        [(ansatz_meas, best_optimization.x)],
        shots=shots
    ).result()[0].data.meas.get_counts()

    records = []

    for bitstring, count in samples.items():

        x = decode_bitstring(bitstring, n)

        selected = int(x.sum())

        feasible = selected == K

        financial_value = portfolio_objective(x)

        records.append({
            "bitstring": bitstring,
            "x": x,
            "count": int(count),
            "probability": count / shots,
            "selected": selected,
            "feasible": feasible,
            "financial_value": financial_value
        })

    total_shots = sum(r["count"] for r in records)

    if total_shots != shots:
        raise RuntimeError("Sample count does not match requested shots.")

    feasible_records = [
        r for r in records if r["feasible"]
    ]

    feasible_shots = sum(
        r["count"] for r in feasible_records
    )

    feasibility_rate = feasible_shots / total_shots

    # With an exact XY circuit this should be 100%.
    if feasibility_rate < 1.0 - 1e-12:
        raise RuntimeError(
            f"Unexpected infeasible samples: {1-feasibility_rate:.4%}. "
            "Check the mixer and bitstring decoding."
        )

    # Probability of obtaining any globally optimal portfolio.
    optimal_shots = sum(
        r["count"]
        for r in feasible_records
        if any(
            np.array_equal(r["x"], solution)
            for solution in optimal_solutions
        )
    )

    optimal_probability = optimal_shots / total_shots

    optimal_given_feasible = (
        optimal_shots / feasible_shots
        if feasible_shots > 0
        else np.nan
    )

    if feasible_shots > 0:

        mean_feasible_objective = sum(
            r["financial_value"] * r["count"]
            for r in feasible_records
        ) / feasible_shots

    else:
        mean_feasible_objective = np.nan

    mean_feasible_ratio = approximation_ratio(
        mean_feasible_objective
    )

    best_feasible = max(
        feasible_records,
        key=lambda r: r["financial_value"],
        default=None
    )

    most_frequent = max(
        records,
        key=lambda r: r["count"],
        default=None
    )

    runtime = time.perf_counter() - start_time

    return {
        "p": p,
        "seed": seed,
        "best_feasible": best_feasible,
        "most_frequent": most_frequent,
        "feasibility_rate": feasibility_rate,
        "optimal_solution_probability": optimal_probability,
        "optimal_probability_given_feasible": optimal_given_feasible,
        "mean_feasible_objective": mean_feasible_objective,
        "mean_feasible_approximation_ratio": mean_feasible_ratio,
        "optimizer_energy": float(best_optimization.fun),
        "optimizer_success": bool(best_optimization.success),
        "optimizer_message": str(best_optimization.message),
        "nfev": int(best_optimization.nfev),
        "optimal_params": np.asarray(best_optimization.x),
        "runtime_seconds": runtime,
        "samples": samples,
        "records": records,
        "feasible_shots": feasible_shots,
        "optimal_shots": optimal_shots
    }


# ============================================================
# 8. RUN CACHE
# ============================================================

RUN_CACHE = {}


def get_cached_run(p, seed, shots=SHOTS, maxiter=MAX_ITER):

    key = (
        int(p),
        int(seed),
        int(shots),
        int(maxiter),
        int(RESTARTS)
    )

    if key not in RUN_CACHE:

        print(
            f"\nRunning XY-QAOA | p={p} | seed={seed} "
            f"| shots={shots}"
        )

        RUN_CACHE[key] = run_xy_qaoa(
            p=p,
            seed=seed,
            shots=shots,
            maxiter=maxiter,
            restarts=RESTARTS
        )

    return RUN_CACHE[key]


# ============================================================
# 9. MAIN INTERMEDIATE EXPERIMENT
# ============================================================

print("\n" + "=" * 65)
print("MAIN XY-QAOA EXPERIMENT")
print("=" * 65)

main_result = get_cached_run(
    p=P_MAIN,
    seed=MAIN_SEED
)

best = main_result["best_feasible"]

if best is None:
    raise RuntimeError("XY-QAOA did not return a feasible portfolio.")

print("Depth:", P_MAIN)
print("Feasibility:", f"{main_result['feasibility_rate']:.2%}")
print(
    "Optimal probability:",
    f"{main_result['optimal_solution_probability']:.2%}"
)
print("Best objective:", best["financial_value"])
print("Exact optimum:", optimal_value)
print("Approximation ratio:", approximation_ratio(best["financial_value"]))
print("Assets:", [
    tickers[i] for i in range(n) if best["x"][i] == 1
])


# ============================================================
# 10. EXPERIMENT: VARY QAOA DEPTH AND RANDOM SEED
# ============================================================


# ============================================================
# 10.1. DISK CHECKPOINT
# ============================================================

CHECKPOINT_PATH = OUTPUT_DIR / (
    f"checkpoint_{DATA_PATH.stem}_"
    f"K{K}_q{Q_RISK:g}_"
    f"shots{SHOTS}_iter{MAX_ITER}_"
    f"r{RESTARTS}.csv"
)

if CHECKPOINT_PATH.exists():
    checkpoint_df = pd.read_csv(CHECKPOINT_PATH)
    raw_results = checkpoint_df.to_dict("records")
    print("Previous results loaded:", len(raw_results))
else:
    raw_results = []

completed_runs = {
    (int(row["p"]), int(row["seed"]))
    for row in raw_results
}

print("Completed configurations:", len(completed_runs))



print("\n" + "=" * 65)
print("INTERMEDIATE EXPERIMENT: DEPTH AND SEEDS")
print("=" * 65)


for p in DEPTHS:

    for seed in EXPERIMENT_SEEDS:

        if (p, seed) in completed_runs:
            print(f"Skipping completed run: p={p}, seed={seed}")
            continue

        result = get_cached_run(
            p=p,
            seed=seed
        )

        best = result["best_feasible"]

        raw_results.append({

            "p": p,
            "seed": seed,

            "best_objective": best["financial_value"],

            "best_sampled_approximation_ratio":
                approximation_ratio(best["financial_value"]),

            "mean_feasible_objective":
                result["mean_feasible_objective"],

            "mean_feasible_approximation_ratio":
                result["mean_feasible_approximation_ratio"],

            "feasibility_rate":
                result["feasibility_rate"],

            "optimal_solution_probability":
                result["optimal_solution_probability"],

            "optimal_probability_given_feasible":
                result["optimal_probability_given_feasible"],

            "nfev": result["nfev"],

            "optimizer_success": result["optimizer_success"],

            "runtime_seconds": result["runtime_seconds"],

            "has_feasible": best is not None,

            "best_portfolio": json.dumps([
                tickers[i]
                for i in range(n)
                if best["x"][i] == 1
            ])

        })

        print(
            f"p={p} | seed={seed} | "
            f"best ratio={approximation_ratio(best['financial_value']):.4f} | "
            f"optimal probability="
            f"{result['optimal_solution_probability']:.2%}"
        )


# ============================================================
# 11. SAVE AND AGGREGATE RESULTS
# ============================================================

raw_df = pd.DataFrame(raw_results)

raw_df.to_csv(
    OUTPUT_DIR / "xy_qaoa_raw_results.csv",
    index=False
)

summary = (
    raw_df
    .groupby("p")
    .agg(
        best_sampled_ratio_mean=(
            "best_sampled_approximation_ratio", "mean"
        ),
        best_sampled_ratio_std=(
            "best_sampled_approximation_ratio", "std"
        ),
        feasible_ratio_mean=(
            "mean_feasible_approximation_ratio", "mean"
        ),
        feasible_ratio_std=(
            "mean_feasible_approximation_ratio", "std"
        ),
        feasibility_mean=(
            "feasibility_rate", "mean"
        ),
        feasibility_std=(
            "feasibility_rate", "std"
        ),
        optimal_probability_mean=(
            "optimal_solution_probability", "mean"
        ),
        optimal_probability_std=(
            "optimal_solution_probability", "std"
        ),
        optimal_given_feasible_mean=(
            "optimal_probability_given_feasible", "mean"
        ),
        n_runs=("seed", "count"),
        n_runs_with_feasible=("has_feasible", "sum"),
        optimizer_success_rate=("optimizer_success", "mean"),
        nfev_mean=("nfev", "mean"),
        runtime_mean=("runtime_seconds", "mean")
    )
    .reset_index()
)

summary.to_csv(
    OUTPUT_DIR / "xy_qaoa_summary.csv",
    index=False
)

print("\nXY-QAOA SUMMARY")
print(summary.to_string(index=False))



# ============================================================
# ROBUST BASELINE COMPARISON
# ============================================================

print("\nQAOA PERFORMANCE BY DEPTH")

performance_by_depth = (
    raw_df.groupby("p")
    .agg(
        qaoa_mean_objective=("best_objective", "mean"),
        qaoa_std_objective=("best_objective", "std"),
        qaoa_best_objective=("best_objective", "max"),
        runs_beating_greedy=(
            "best_objective",
            lambda values: (values > greedy_value + 1e-12).sum()
        ),
        runs_beating_local_search=(
            "best_objective",
            lambda values: (values > local_value + 1e-12).sum()
        ),
        total_runs=("seed", "count")
    )
    .reset_index()
)

performance_by_depth["fraction_beating_greedy"] = (
    performance_by_depth["runs_beating_greedy"]
    / performance_by_depth["total_runs"]
)

performance_by_depth["fraction_beating_local_search"] = (
    performance_by_depth["runs_beating_local_search"]
    / performance_by_depth["total_runs"]
)

performance_by_depth.to_csv(
    OUTPUT_DIR / "baseline_comparison_by_depth.csv",
    index=False
)

print(performance_by_depth.to_string(index=False))

print("\nCLASSICAL BASELINE VALUES")
print("Greedy:", greedy_value)
print("Greedy + Local Search:", local_value)
print("Exact optimum:", optimal_value)

print("\nPROBLEM SIZE AND CONSTRAINT ANALYSIS")

total_binary_solutions = 2 ** n
feasible_portfolio_count = len(feasible_x)

print("Number of assets:", n)
print("Assets selected:", K)
print("Total binary solutions:", total_binary_solutions)
print("Feasible portfolios:", feasible_portfolio_count)

print(
    "Feasible search-space proportion:",
    f"{100 * feasible_portfolio_count / total_binary_solutions:.2f}%"
)


Intermediate assets: 16
Assets to select: 6
Data file: parametros_16_activos.json

Minimum covariance eigenvalue: 0.014277364817434017
Financial data validation passed.

CARDINALITY SENSITIVITY ANALYSIS
 K  feasible_portfolios  feasible_percentage  greedy_objective  exact_objective    greedy_gap  greedy_ratio
 2                  120             0.183105          1.581663         1.581663  0.000000e+00           1.0
 3                  560             0.854492          2.060529         2.060529  0.000000e+00           1.0
 4                 1820             2.777100          2.529029         2.529029  0.000000e+00           1.0
 5                 4368             6.665039          2.863240         2.863240  4.440892e-16           1.0
 6                 8008            12.219238          3.167704         3.167704  0.000000e+00           1.0
 7                11440            17.456055          3.339591         3.339591  0.000000e+00           1.0
 8                12870            19.638

In [ ]:

# ============================================================
# 13. FINAL INTERMEDIATE VISUALIZATIONS
# ============================================================


plt.figure(figsize=(9, 6))

plt.errorbar(
    summary["p"],
    summary["feasible_ratio_mean"],
    yerr=summary["feasible_ratio_std"].fillna(0),
    marker="o",
    capsize=4,
    label="Mean feasible approximation ratio"
)

plt.plot(
    summary["p"],
    summary["best_sampled_ratio_mean"],
    marker="s",
    linestyle="--",
    label="Best sampled approximation ratio"
)

# Classical references.
if optimal_value > 0:
    plt.axhline(
        greedy_value / optimal_value,
        linestyle=":",
        label="Greedy = Greedy + Local Search = Exact optimum"
        if np.isclose(greedy_value, local_value)
        and np.isclose(local_value, optimal_value)
        else "Greedy baseline"
    )

plt.xlabel("QAOA depth (p)")
plt.ylabel("Approximation ratio")
plt.title("XY-QAOA solution quality vs circuit depth")
plt.xticks(DEPTHS)
plt.ylim(0, 1.05)
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()

plt.savefig(
    OUTPUT_DIR / "plot_1_solution_quality.png",
    dpi=300,
    bbox_inches="tight"
)
plt.close()


# ------------------------------------------------------------
# 16.2 OPTIMAL SOLUTION PROBABILITY VS DEPTH
# ------------------------------------------------------------

plt.figure(figsize=(9, 6))

plt.errorbar(
    summary["p"],
    summary["optimal_probability_mean"] * 100,
    yerr=summary["optimal_probability_std"].fillna(0) * 100,
    marker="o",
    capsize=4
)

plt.xlabel("QAOA depth (p)")
plt.ylabel("Optimal solution probability (%)")
plt.title("Probability of sampling the global optimum")
plt.xticks(DEPTHS)
plt.ylim(0, 105)
plt.grid(alpha=0.3)
plt.tight_layout()

plt.savefig(
    OUTPUT_DIR / "plot_2_optimal_probability.png",
    dpi=300,
    bbox_inches="tight"
)
plt.close()


# ------------------------------------------------------------
# 16.3 RUNTIME VS DEPTH
# ------------------------------------------------------------

plt.figure(figsize=(9, 6))

plt.plot(
    summary["p"],
    summary["runtime_mean"],
    marker="o"
)

plt.xlabel("QAOA depth (p)")
plt.ylabel("Mean runtime per execution (seconds)")
plt.title("Computational cost vs QAOA depth")
plt.xticks(DEPTHS)
plt.grid(alpha=0.3)
plt.tight_layout()

plt.savefig(
    OUTPUT_DIR / "plot_3_runtime.png",
    dpi=300,
    bbox_inches="tight"
)
plt.close()


# ------------------------------------------------------------
# 16.4 GREEDY QUALITY VS CARDINALITY K
# ------------------------------------------------------------

plt.figure(figsize=(9, 6))

plt.plot(
    k_df["K"],
    k_df["greedy_ratio"],
    marker="o",
    label="Greedy approximation ratio"
)

plt.axhline(
    1.0,
    linestyle="--",
    label="Exact optimum"
)

plt.xlabel("Number of selected assets (K)")
plt.ylabel("Greedy approximation ratio")
plt.title("Classical greedy performance vs cardinality")
plt.xticks(k_df["K"])
plt.ylim(0, 1.05)
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()

plt.savefig(
    OUTPUT_DIR / "plot_4_greedy_vs_cardinality.png",
    dpi=300,
    bbox_inches="tight"
)
plt.close()


# ------------------------------------------------------------
# 16.5 FEASIBLE SEARCH SPACE VS K
# ------------------------------------------------------------

plt.figure(figsize=(9, 6))

plt.bar(
    k_df["K"].astype(str),
    k_df["feasible_percentage"]
)

plt.xlabel("Number of selected assets (K)")
plt.ylabel("Feasible portfolios (%)")
plt.title("Feasible search-space proportion vs cardinality")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()

plt.savefig(
    OUTPUT_DIR / "plot_5_feasible_space_vs_cardinality.png",
    dpi=300,
    bbox_inches="tight"
)
plt.close()

print("\nAll final plots have been saved in:", OUTPUT_DIR.resolve())
